# Moon synthetics from the RSGT catalogue — Julia

The catalogue holds, for each of 1000 lunar 1-D models, the **receiver-side strain Green's
tensors** (computed with flexOPT.jl, OPT3, 0–0.05 Hz, elastic). With reciprocity, they give the
seismograms at any station for any event (position, depth down to the top of the core, moment
tensor or force) in a fraction of a second, with no solver.

You need the folder of `rsgt_XXXXX.h5` files (one per model, ≈ 0.6 GB each) and, in the same
folder as this notebook, `moon_rsgt_h5.jl`.

Conventions: angles in degrees, depth in km, moment tensor in the GCMT convention
(r = up, t = south, p = east) in N·m, force (f_r, f_t, f_p) in N. Components Z (up), N, E,
R (horizontal, away from the event), T. Output: velocity (m/s) by default.

Packages: `JLD2` (reading) and `CairoMakie` (plots); both are in the flexOPT.jl environment.

## 1. Where is the catalogue?

Change the path below. If files are still being copied, only use the ones that are complete.

In [ ]:
include("moon_rsgt_h5.jl")
using CairoMakie

MOON_H5_DIR[] = expanduser("~/Dropbox/stage_tutore_2026/catalogue_h5")
ids = moonH5Ids()
println(length(ids), " models: ", first(ids, 5), " ...")

## 2. What is inside one file?

In [ ]:
mid = ids[1]
jldopen(moonH5File(mid)) do f
    for k in keys(f)
        v = f[k]
        println(rpad(k, 20), v isa AbstractArray ? (size(v), eltype(v)) : v)
    end
end

## 3. One event, one station (Z, R, T velocity)

In [ ]:
station = (stla=-3.0, stlo=-23.4)                        # near Apollo 12
event = (evla=20.0, evlo=15.0, evdp=900.0)               # a deep moonquake
M = moonH5DoubleCouple(30, 60, 90; M0=1e15)              # N·m

@time t, tr = moonH5Synthetics(mid; station..., event..., M...)     # first call loads the model (~0.6 GB)
@time t, tr = moonH5Synthetics(mid; station..., event..., M...)     # then it is fast
println("distance ", round(moonH5Distance(event.evla, event.evlo, station.stla, station.stlo), digits=2), "°")
fig = Figure(size=(900, 600))
for (i, c) in enumerate("ZRT")
    ax = Axis(fig[i, 1]; ylabel="$c (m/s)", xlabel=i == 3 ? "time (s)" : "", title=i == 1 ? "model $mid" : "")
    lines!(ax, t, tr[string(c)]; color=:black, linewidth=0.8)
end
fig

## 4. Record section (stations every 10° along one azimuth)

In [ ]:
ev = (evla=0.0, evlo=0.0, evdp=300.0)
fig = Figure(size=(900, 800))
ax = Axis(fig[1, 1]; xlabel="time (s)", ylabel="epicentral distance (°)",
          title="model $mid, Z velocity, event at $(ev.evdp) km (each trace normalised)")
for Δ in 5:10:175
    stla, stlo = moonH5PointAt(ev.evla, ev.evlo, Δ, 45.0)
    t, tr = moonH5Synthetics(mid; stla=stla, stlo=stlo, ev..., M..., comps="Z")
    z = tr["Z"] ./ maximum(abs, tr["Z"])
    lines!(ax, t, Δ .+ 4 .* z; color=:black, linewidth=0.6)
end
fig

## 5. The same event and station for several models

In [ ]:
fig = Figure(size=(900, 400))
ax = Axis(fig[1, 1]; xlabel="time (s)", ylabel="Z velocity (m/s)")
for m in first(ids, 5)
    t, tr = moonH5Synthetics(m; station..., event..., M..., comps="Z")
    lines!(ax, t, tr["Z"]; linewidth=0.8, label=m)
end
axislegend(ax)
fig

## 6. An impact: vertical force at the surface

In [ ]:
t, tr = moonH5Synthetics(mid; station..., evla=5.0, evlo=40.0, evdp=0.0, force=(-1e10, 0.0, 0.0))
fig = Figure(size=(900, 600))
for (i, c) in enumerate("ZRT")
    ax = Axis(fig[i, 1]; ylabel="$c (m/s)")
    lines!(ax, t, tr[string(c)]; color=:black, linewidth=0.8)
end
fig

Options: `comps="ZNE"`, `quantity="displacement"` or `"acceleration"`, `dt=0.5`, `half_duration=15.0` (s, keep ≥ 10 s: the catalogue stops at 0.05 Hz).